In [0]:
# Import required libraries
from pyspark.sql import functions as F

# Azure SQL connection details
server_name = "capstone09.database.windows.net"
database_name = "capstone09"

jdbc_url = (
    f"jdbc:sqlserver://{server_name}:1433;"
    f"databaseName={database_name};"
    "encrypt=true;"
    "trustServerCertificate=false;"
    "hostNameInCertificate=*.database.windows.net;"
    "loginTimeout=30;"
)

# Get credentials securely
username = dbutils.secrets.get(
    scope="azure-sql-scope",
    key="sql-user"
)

password = dbutils.secrets.get(
    scope="azure-sql-scope",
    key="sql-pwd"
)

connection_properties = {
    "user": username,
    "password": password,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

print("JDBC configuration completed")

In [0]:

from pyspark.sql import functions as F
from pyspark.sql.types import *
import json

# --------------------------------------------------
# 1. SELECT EXISTING CATALOG AND SCHEMA
# --------------------------------------------------

spark.sql("USE CATALOG capstone9_smartmeter")
spark.sql("USE SCHEMA smart_grid_bronze")

# --------------------------------------------------
# 2. DEFINE SOURCE, SCHEMA AND CHECKPOINT PATHS
# --------------------------------------------------

base_path = "/Volumes/capstone9_smartmeter/smart_grid_bronze/raw_volume"

schema_base = f"{base_path}/_schemas/smart_grid_new"
checkpoint_base = f"{base_path}/_checkpoints/smart_grid_new"  # existing checkpoints used to build the live Bronze tables

# --------------------------------------------------
# 3. DEFINE EXPLICIT SCHEMAS
# --------------------------------------------------

readings_schema = StructType([
    StructField("reading_id", StringType(), True),
    StructField("meter_id", StringType(), True),
    StructField("substation_id", StringType(), True),
    StructField("interval_start_ts", TimestampType(), True),
    StructField("interval_end_ts", TimestampType(), True),
    StructField("energy_kwh", DoubleType(), True),
    StructField("voltage_v", DoubleType(), True),
    StructField("current_a", DoubleType(), True),
    StructField("power_factor", DoubleType(), True),
    StructField("quality_code", StringType(), True),
    StructField("reading_source", StringType(), True),
    StructField("received_ts", TimestampType(), True)
])

outage_schema = StructType([
    StructField("outage_id", StringType(), True),
    StructField("substation_id", StringType(), True),
    StructField("outage_start_ts", TimestampType(), True),
    StructField("outage_end_ts", TimestampType(), True),
    StructField("outage_cause", StringType(), True),
    StructField("severity", StringType(), True),
    StructField("affected_meters", IntegerType(), True),
    StructField("incident_status", StringType(), True),
    StructField("reported_ts", TimestampType(), True),
    StructField("repair_team", StringType(), True)
])

recovery_schema = StructType([
    StructField("recovery_event_id", StringType(), True),
    StructField("meter_id", StringType(), True),
    StructField("outage_id", StringType(), True),
    StructField("event_ts", TimestampType(), True),
    StructField("recovery_type", StringType(), True),
    StructField("signal_strength", DoubleType(), True),
    StructField("firmware_version", StringType(), True),
    StructField("acknowledged_flag", StringType(), True),  # source sends Y/N; BOOLEAN pushed every value into _rescued_data
    StructField("technician_id", StringType(), True),
    StructField("received_ts", TimestampType(), True)
])

# --------------------------------------------------
# 4. AUTO LOADER - SMART METER READINGS
# --------------------------------------------------

readings_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", f"{schema_base}/readings")
    .option("rescuedDataColumn", "_rescued_data")
    .schema(readings_schema)
    .load(f"{base_path}/telemetry_data")
)

# --------------------------------------------------
# 5. AUTO LOADER - GRID OUTAGE EVENTS
# --------------------------------------------------

outage_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("pathGlobFilter", "grid_outage_events.csv")
    .option("cloudFiles.schemaLocation", f"{schema_base}/outages")
    .option("rescuedDataColumn", "_rescued_data")
    .schema(outage_schema)
    .load(f"{base_path}/event_data")
)

# --------------------------------------------------
# 6. AUTO LOADER - METER RECOVERY EVENTS
# --------------------------------------------------

recovery_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("pathGlobFilter", "meter_recovery_events.csv")
    .option("cloudFiles.schemaLocation", f"{schema_base}/recovery")
    .option("rescuedDataColumn", "_rescued_data")
    .schema(recovery_schema)
    .load(f"{base_path}/event_data")
)

# --------------------------------------------------
# 7. ADD INGESTION METADATA
# --------------------------------------------------

readings_df = (
    readings_df
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
)

outage_df = (
    outage_df
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
)

recovery_df = (
    recovery_df
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
)

# --------------------------------------------------
# 8. WRITE TO BRONZE DELTA TABLES (append-only, replay-safe)
# --------------------------------------------------

def assert_checkpoint_continuity(folder, table_name):
    # Refuse to start if the checkpoint is missing or does not belong to the stream that wrote the table
    cp = f"{checkpoint_base}/{folder}"
    try:
        cp_query_id = json.loads(dbutils.fs.head(f"{cp}/metadata"))["id"]
        commits = [int(f.name.strip("/")) for f in dbutils.fs.ls(f"{cp}/commits") if f.name.strip("/").isdigit()]
    except Exception as e:
        raise RuntimeError(f"Checkpoint {cp} missing or unreadable - not starting (would replay all source files)") from e
    last = (
        spark.sql(f"DESCRIBE HISTORY capstone9_smartmeter.smart_grid_bronze.{table_name}")
        .filter("operation = 'STREAMING UPDATE'")
        .orderBy(F.desc("version"))
        .select(F.col("operationParameters.queryId").alias("query_id"),
                F.col("operationParameters.epochId").cast("int").alias("epoch_id"))
        .first()
    )
    if last is None or last.query_id != cp_query_id or last.epoch_id > max(commits, default=-1):
        raise RuntimeError(f"Checkpoint {cp} does not match {table_name} history - not starting")
    print(f"{table_name}: checkpoint OK (query {cp_query_id}, last committed batch {max(commits)})")

def start_stream(df, folder, table_name):
    # Same Delta append sink that built the live tables: no overwrite, no schema change
    return (
        df.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", f"{checkpoint_base}/{folder}")
        .trigger(availableNow=True)
        .toTable(f"capstone9_smartmeter.smart_grid_bronze.{table_name}")
    )

# (stream, checkpoint folder, table)
# Recovery events use the recovery_v2 checkpoint (acknowledged_flag migrated to STRING on 2026-11-04)
streams = [
    (readings_df, "readings", "bronze_smart_meter_readings"),
    (outage_df, "outages", "bronze_grid_outage_events"),
    (recovery_df, "recovery_v2", "bronze_meter_recovery_events"),
]

# Verify every checkpoint before starting any stream (a lost checkpoint would replay all files)
for _, folder, table_name in streams:
    assert_checkpoint_continuity(folder, table_name)

before = {t: spark.table(t).count() for _, _, t in streams}
queries = [(t, start_stream(df, folder, t)) for df, folder, t in streams]

# Wait for all availableNow streams to finish before validating
for _, q in queries:
    q.awaitTermination()

# --------------------------------------------------
# 9. VALIDATE: replay behaviour and rescued data
# --------------------------------------------------

for t, _ in queries:
    s = spark.table(t).agg(F.count("*").alias("rows"), F.count_if(F.col("_rescued_data").isNotNull()).alias("rescued")).first()
    print(f"{t}: rows before={before[t]}, after={s.rows}, added={s.rows - before[t]}, rescued rows={s.rescued}")

print("ADLS AUTO LOADER BRONZE INGESTION COMPLETED")


In [0]:
spark.sql("""
SELECT 'Smart Meter Readings' AS dataset, COUNT(*) AS records
FROM capstone9_smartmeter.smart_grid_bronze.bronze_smart_meter_readings

UNION ALL

SELECT 'Grid Outage Events', COUNT(*)
FROM capstone9_smartmeter.smart_grid_bronze.bronze_grid_outage_events

UNION ALL

SELECT 'Meter Recovery Events', COUNT(*)
FROM capstone9_smartmeter.smart_grid_bronze.bronze_meter_recovery_events
""").show()

In [0]:

from pyspark.sql import functions as F

# 1. Read the three Azure SQL tables

meter_df = spark.read.jdbc(
    url=jdbc_url,
    table="dbo.meter_master",
    properties=connection_properties
)

substation_df = spark.read.jdbc(
    url=jdbc_url,
    table="dbo.substation_master",
    properties=connection_properties
)

tariff_df = spark.read.jdbc(
    url=jdbc_url,
    table="dbo.tariff_lookup",
    properties=connection_properties
)

print("All three Azure SQL tables read successfully!")

# 2. Select existing catalog and schema

spark.sql("USE CATALOG capstone9_smartmeter")
spark.sql("USE SCHEMA smart_grid_bronze")

# 2b. Typed contract for numeric and date columns
#     The Azure SQL extract delivers every column as text. Cast to the Handbook types on the way in.
#     Casting is strict: if any non-empty source value cannot be cast, the run fails instead of
#     writing NULL, so Bronze never silently loses a source value.

TYPED_COLUMNS = {
    "bronze_meter_master": {"installation_date": "DATE", "rated_voltage": "DOUBLE"},
    "bronze_substation_master": {"capacity_mva": "DOUBLE", "voltage_level_kv": "DOUBLE",
                                 "commissioned_date": "DATE", "latitude": "DOUBLE", "longitude": "DOUBLE"},
    "bronze_tariff_lookup": {"rate_per_kwh": "DOUBLE", "effective_date": "DATE",
                             "expiry_date": "DATE", "peak_multiplier": "DOUBLE"},
}

# Leading/trailing spaces and control characters (e.g. trailing \r from CRLF extracts), non-breaking space
CTRL_WS = r"^[\x00-\x20\xA0]+|[\x00-\x20\xA0]+$"

def cast_typed(df, types, label):
    for c in types:
        v = F.regexp_replace(F.col(c).cast("string"), CTRL_WS, "")
        df = df.withColumn(f"__raw_{c}", F.when(v != "", v))
    df = df.withColumns({c: F.expr(f"try_cast(`__raw_{c}` AS {t})") for c, t in types.items()})
    failed = df.agg(*[F.count_if(F.col(f"__raw_{c}").isNotNull() & F.col(c).isNull()).alias(c) for c in types]).first().asDict()
    failed = {c: n for c, n in failed.items() if n}
    if failed:
        raise ValueError(f"{label}: values that cannot be cast to the contract type {failed} - nothing written")
    return df.drop(*[f"__raw_{c}" for c in types])

# 3. Controlled incremental snapshot ingestion (append-only, replay-safe)
#    A source row is appended only if an identical row (same values in every source column)
#    is not already in Bronze. Re-running with unchanged source adds 0 rows; a changed row is
#    appended as a new version, so earlier snapshots are preserved. Latest version = max _ingested_at per key.

datasets = [
    (meter_df, "bronze_meter_master"),
    (substation_df, "bronze_substation_master"),
    (tariff_df, "bronze_tariff_lookup")
]

row_hash = lambda cols: F.sha2(F.to_json(F.struct(*cols)), 256)

for df, table_name in datasets:
    types = TYPED_COLUMNS[table_name]
    df = cast_typed(df, types, f"{table_name} (Azure SQL source)")
    source_cols = df.columns
    snapshot = df.withColumn("_row_hash", row_hash(source_cols))

    if spark.catalog.tableExists(table_name):
        existing = spark.table(table_name).select(row_hash(source_cols).alias("_row_hash")).distinct()
        snapshot = snapshot.join(existing, "_row_hash", "left_anti")

    new_rows = (
        snapshot.drop("_row_hash")
        .withColumn("_source", F.lit("Azure SQL"))
        .withColumn("_ingested_at", F.current_timestamp())
    )
    added = new_rows.count()
    if added:
        new_rows.write.format("delta").mode("append").saveAsTable(table_name)

    print(f"{table_name}: {added} new or changed rows appended")

# 4. Validate all three tables

for _, table_name in datasets:
    full_table_name = (
        f"capstone9_smartmeter.smart_grid_bronze.{table_name}"
    )

    t = spark.table(full_table_name)
    dtypes = dict(t.dtypes)
    print(
        table_name,
        "Records:",
        t.count(),
        "| types:",
        {c: dtypes[c] for c in TYPED_COLUMNS[table_name]}
    )

print("AZURE SQL TO BRONZE INGESTION COMPLETED!")


## Source data profiling

Read-only profiling of the six Bronze tables against the Dataset Handbook contracts (§3), DQ scope (§4) and EDA tasks (§5). Nothing is written. Values are observed, not assumed defect counts.

* P1 setup and cleaned views (trailing `\r` stripped, empty text treated as NULL, `acknowledged_flag` recovered from `_rescued_data`)
* P2 record counts, duplicates and raw-format issues
* P3 missing values
* P4 invalid values, types, flags and categories
* P5 timestamp ranges
* P6 percentiles and outliers (3 x IQR fence)
* P7 key coverage and orphans
* P8 business rules and interval completeness

In [0]:
from functools import reduce
from pyspark.sql import DataFrame, functions as F

BRONZE = "capstone9_smartmeter.smart_grid_bronze"
META = {"_source", "_source_file", "_ingested_at", "_rescued_data"}

SOURCES = {
    "readings": ("bronze_smart_meter_readings", "reading_id"),
    "outages": ("bronze_grid_outage_events", "outage_id"),
    "recovery": ("bronze_meter_recovery_events", "recovery_event_id"),
    "meters": ("bronze_meter_master", "meter_id"),
    "substations": ("bronze_substation_master", "substation_id"),
    "tariffs": ("bronze_tariff_lookup", "tariff_code"),
}
FOREIGN_KEYS = [
    ("readings", "meter_id", "meters", "meter_id"),
    ("readings", "substation_id", "substations", "substation_id"),
    ("outages", "substation_id", "substations", "substation_id"),
    ("recovery", "meter_id", "meters", "meter_id"),
    ("recovery", "outage_id", "outages", "outage_id"),
    ("meters", "substation_id", "substations", "substation_id"),
    ("meters", "tariff_code", "tariffs", "tariff_code"),
]
NUMERIC = {
    "readings": ["energy_kwh", "voltage_v", "current_a", "power_factor"],
    "outages": ["affected_meters"],
    "recovery": ["signal_strength"],
    "meters": ["rated_voltage"],
    "substations": ["capacity_mva", "voltage_level_kv", "latitude", "longitude"],
    "tariffs": ["rate_per_kwh", "peak_multiplier"],
}
TIMESTAMPS = {
    "readings": ["interval_start_ts", "interval_end_ts", "received_ts"],
    "outages": ["outage_start_ts", "outage_end_ts", "reported_ts"],
    "recovery": ["event_ts", "received_ts"],
    "meters": ["installation_date"],
    "substations": ["commissioned_date"],
    "tariffs": ["effective_date", "expiry_date"],
}
CATEGORIES = {
    "readings": ["quality_code", "reading_source"],
    "outages": ["incident_status", "severity", "outage_cause", "repair_team"],
    "recovery": ["recovery_type", "acknowledged_flag", "firmware_version"],
    "meters": ["meter_type", "meter_status", "customer_class", "service_region", "manufacturer", "tariff_code"],
    "substations": ["region", "operator", "active_flag"],
    "tariffs": ["customer_class", "currency", "active_flag", "regulatory_zone"],
}
FLAGS = {"active_flag", "acknowledged_flag"}

raw = {n: spark.table(f"{BRONZE}.{t}") for n, (t, _) in SOURCES.items()}
schemas = {n: dict(df.dtypes) for n, df in raw.items()}
columns = {n: [c for c in s if c not in META] for n, s in schemas.items()}


def clean(n):
    exprs = []
    for c in columns[n]:
        if c == "acknowledged_flag":
            v = F.coalesce(F.col(c).cast("string"), F.get_json_object("_rescued_data", f"$.{c}"))
        elif schemas[n][c] == "string":
            v = F.trim(F.regexp_replace(c, "\r", ""))
        else:
            exprs.append(F.col(c))
            continue
        exprs.append(F.when(v == "", None).otherwise(v).alias(c))
    return raw[n].select(exprs)


dfs = {n: clean(n) for n in SOURCES}

In [0]:
overview = []
for n, (table, key) in SOURCES.items():
    df, strings = dfs[n], [c for c in columns[n] if schemas[n][c] == "string"]
    a = df.agg(F.count("*"), F.countDistinct(key), sum(F.count_if(F.col(c).isNull()) for c in columns[n])).first()
    b = raw[n].agg(
        F.count_if(F.col("_rescued_data").isNotNull()) if "_rescued_data" in schemas[n] else F.lit(0),
        F.count_if(reduce(lambda x, y: x | y, [F.col(c).contains("\r") for c in strings])),
    ).first()
    overview.append((n, table, a[0], len(columns[n]), a[1], a[0] - a[1], a[0] - df.distinct().count(), a[2], b[0], b[1]))

display(spark.createDataFrame(overview, "dataset string, bronze_table string, rows long, columns int, unique_keys long, "
    "duplicate_key_rows long, exact_duplicate_rows long, null_cells long, rescued_rows long, rows_with_carriage_return long"))

batch = F.element_at(F.split("_source_file", "/"), -1)
display(
    raw["readings"].groupBy("reading_id")
    .agg(F.count("*").alias("copies"), F.array_sort(F.collect_set(batch)).alias("batches"))
    .filter("copies > 1")
    .groupBy(F.when(F.size("batches") > 1, "re-delivered in a later batch").otherwise("repeated inside one batch").alias("duplicate_type"), "batches")
    .agg(F.count("*").alias("reading_ids"), F.sum(F.col("copies") - 1).alias("extra_rows"))
    .orderBy("duplicate_type", "batches")
)

In [0]:
key_columns = {k for _, k in SOURCES.values()} | {fk for _, fk, _, _ in FOREIGN_KEYS}
rows_by_dataset = {o[0]: o[2] for o in overview}

nulls = []
for n, df in dfs.items():
    counts = df.agg(*[F.count_if(F.col(c).isNull()).alias(c) for c in columns[n]]).first().asDict()
    nulls += [(n, c, "key" if c in key_columns else "optional attribute", v, round(v / rows_by_dataset[n] * 100, 3))
              for c, v in counts.items() if v]

display(spark.createDataFrame(nulls, "dataset string, column string, role string, missing long, missing_pct double"))

In [0]:
def invalid(n, c):
    v = F.col(c)
    if c in FLAGS:
        return F.count_if(v.isNotNull() & ~v.isin("Y", "N"))
    target = "timestamp" if c in TIMESTAMPS[n] else "double"
    return F.count_if(v.isNotNull() & F.expr(f"try_cast({c} AS {target})").isNull())

type_checks = []
for n in SOURCES:
    cols = NUMERIC[n] + TIMESTAMPS[n] + [f for f in FLAGS if f in columns[n]]
    r = dfs[n].agg(*[invalid(n, c).alias(c) for c in cols]).first()
    type_checks += [(n, c, schemas[n][c], "Y/N flag" if c in FLAGS else ("timestamp" if c in TIMESTAMPS[n] else "number"), r[c]) for c in cols]

display(spark.createDataFrame(type_checks, "dataset string, column string, bronze_type string, expected string, invalid_values long")
        .orderBy(F.desc("invalid_values"), "dataset"))

display(raw["recovery"].agg(
    F.count_if(F.col("acknowledged_flag").isNull()).alias("bronze_acknowledged_flag_null"),
    F.count_if(F.get_json_object("_rescued_data", "$.acknowledged_flag").isNotNull()).alias("value_only_in_rescued_data"),
))

display(reduce(DataFrame.unionByName, [
    dfs[n].groupBy(F.col(c).alias("value")).count().agg(
        F.lit(n).alias("dataset"), F.lit(c).alias("column"),
        F.count("value").alias("distinct_values"),
        F.sum(F.when(F.col("value").isNull(), F.col("count")).otherwise(0)).alias("nulls"),
        F.concat_ws(", ", F.sort_array(F.collect_list(
            F.concat(F.coalesce("value", F.lit("NULL")), F.lit(" ("), F.col("count").cast("string"), F.lit(")"))))).alias("value_counts"),
    )
    for n, cs in CATEGORIES.items() for c in cs
]))

In [0]:
ranges = []
for n, cs in TIMESTAMPS.items():
    ts = {c: F.expr(f"try_cast({c} AS TIMESTAMP)") for c in cs}
    r = dfs[n].agg(*[f(e).alias(f"{c}_{f.__name__}") for c, e in ts.items() for f in (F.min, F.max)],
                   *[F.countDistinct(F.to_date(e)).alias(f"{c}_days") for c, e in ts.items()]).first()
    ranges += [(n, c, r[f"{c}_min"], r[f"{c}_max"], r[f"{c}_days"]) for c in cs]

display(spark.createDataFrame(ranges, "dataset string, column string, earliest timestamp, latest timestamp, distinct_days long"))

In [0]:
outliers = []
for n, cs in NUMERIC.items():
    df = dfs[n].select([F.expr(f"try_cast({c} AS DOUBLE)").alias(c) for c in cs])
    q = df.agg(*[F.percentile_approx(c, [0.0, 0.01, 0.25, 0.5, 0.75, 0.99, 1.0], 10000).alias(c) for c in cs]).first()
    fences = {c: (q[c][2] - 3 * (q[c][4] - q[c][2]), q[c][4] + 3 * (q[c][4] - q[c][2])) for c in cs}
    out = {c: (F.col(c) < lo) | (F.col(c) > hi) for c, (lo, hi) in fences.items()}
    o = df.agg(*[F.count_if(e).alias(c) for c, e in out.items()],
               *[F.array_sort(F.collect_set(F.when(e, F.col(c)))).cast("string").alias(f"{c}_values") for c, e in out.items()]).first()
    outliers += [(n, c, *q[c], *fences[c], o[c], o[f"{c}_values"]) for c in cs]

display(spark.createDataFrame(outliers,
    "dataset string, column string, min double, p01 double, p25 double, p50 double, p75 double, p99 double, max double, "
    "lower_fence double, upper_fence double, outliers long, outlier_values string"))

In [0]:
coverage = []
for child, ck, parent, pk in FOREIGN_KEYS:
    keys = dfs[child].select(F.col(ck).alias("k"))
    orphans = keys.join(dfs[parent].select(F.col(pk).alias("k")), "k", "left_anti")
    r = orphans.agg(F.count_if(F.col("k").isNull()), F.count_if(F.col("k").isNotNull()),
                    F.slice(F.array_sort(F.collect_set("k")), 1, 5).cast("string")).first()
    total = rows_by_dataset[child]
    coverage.append((f"{child}.{ck}", f"{parent}.{pk}", total, r[0], r[1], round((total - r[0] - r[1]) / total * 100, 2), r[2]))

display(spark.createDataFrame(coverage,
    "foreign_key string, references string, rows long, null_keys long, orphan_rows long, coverage_pct double, sample_orphans string"))

In [0]:
r, o, v, m, t, s = (dfs[n] for n in ["readings", "outages", "recovery", "meters", "tariffs", "substations"])
meter_ref = m.select("meter_id", F.col("substation_id").alias("meter_substation_id"), F.to_date("installation_date").alias("installation_date"))
readings_meter = r.join(meter_ref, "meter_id")
recovery_outage = (v.join(o.select("outage_id", "outage_start_ts", "incident_status", F.col("substation_id").alias("outage_substation_id")), "outage_id")
                    .join(meter_ref, "meter_id"))

rules = [
    ("readings", "Handbook §3 timestamp order", "interval_end_ts <= interval_start_ts", r),
    ("readings", "Handbook §3 interval completeness and units", "unix_timestamp(interval_end_ts) - unix_timestamp(interval_start_ts) <> 300", r),
    ("readings", "Handbook §3 timestamp order", "received_ts < interval_end_ts", r),
    ("readings", "Handbook §4 type/range validation", "energy_kwh < 0", r),
    ("readings", "Handbook §3 relationship coverage", "substation_id <> meter_substation_id", readings_meter),
    ("readings", "Handbook §3 timestamp order", "to_date(interval_start_ts) < installation_date", readings_meter),
    ("outages", "Handbook §4 chronology", "outage_end_ts < outage_start_ts", o),
    ("outages", "Handbook §4 chronology", "reported_ts < outage_start_ts", o),
    ("outages", "Handbook §4 event-type checks", "incident_status IN ('OPEN', 'INVESTIGATING') AND outage_end_ts IS NOT NULL", o),
    ("outages", "Handbook §4 event-type checks", "incident_status = 'RESOLVED' AND outage_end_ts IS NULL", o),
    ("recovery", "Handbook §4 lifecycle ordering", "event_ts < outage_start_ts", recovery_outage),
    ("recovery", "Handbook §4 lifecycle ordering", "received_ts < event_ts", v),
    ("recovery", "Handbook §4 status transition", "incident_status <> 'RESOLVED'", recovery_outage),
    ("recovery", "Handbook §4 references", "meter_substation_id <> outage_substation_id", recovery_outage),
    ("tariffs", "Handbook §3 timestamp order", "to_date(expiry_date) < to_date(effective_date)", t),
    ("substations", "Handbook §3 unusual values", "NOT (try_cast(latitude AS DOUBLE) BETWEEN -90 AND 90 AND try_cast(longitude AS DOUBLE) BETWEEN -180 AND 180)", s),
]

display(spark.createDataFrame(
    [(n, ref, rule, df.filter(rule).count(), df.count()) for n, ref, rule, df in rules],
    "dataset string, reference string, violation string, violating_rows long, rows_checked long"))

unique_readings = r.dropDuplicates()
display(
    unique_readings.groupBy("meter_id").agg(
        F.countDistinct("interval_start_ts").alias("actual"),
        ((F.max("interval_end_ts").cast("long") - F.min("interval_start_ts").cast("long")) / 300).alias("expected"))
    .agg(F.count("*").alias("meters"), F.sum("actual").alias("actual_intervals"), F.sum("expected").cast("long").alias("expected_5min_intervals"))
    .withColumn("missing_intervals", F.col("expected_5min_intervals") - F.col("actual_intervals"))
    .withColumn("missing_interval_pct", F.round(F.col("missing_intervals") / F.col("expected_5min_intervals") * 100, 2))
    .withColumn("same_meter_interval_different_reading_id",
                F.lit(unique_readings.count() - unique_readings.select("meter_id", "interval_start_ts").distinct().count()))
)

## Bronze data quality findings

Observed on 4 Nov 2026 from P1–P8. Severity is a profiling judgement for Silver design; the trainer should confirm it.

| # | Dataset | Problem | Observed | Evidence | Severity |
| --- | --- | --- | --- | --- | --- |
| 1 | readings | Duplicate `reading_id`, all exact copies | 49 extra rows (12,617 rows, 12,568 unique) | P2: 46 re-delivered from batches 001–010 in batch_011, 3 repeated inside batch_011 | High |
| 2 | readings | `energy_kwh` sentinel values | 45 outliers: -999 (16) and 999999.99 (29) | P6: outside 3 x IQR fences (-14.78, 26.8); p01–p99 is 0.21–11.9 | High |
| 3 | readings | Negative energy | 16 rows | P8: same rows as the -999 sentinel | High |
| 4 | readings | Readings before meter `installation_date` | 4,414 rows | P8: reading date earlier than `meter_master.installation_date` (dates span Mar–Oct 2026) | High (confirm) |
| 5 | readings | Missing optional attributes | 13 nulls: `current_a` 5, `power_factor` 3, `reading_source` 3, `voltage_v` 1, `quality_code` 1 | P3: no key or timestamp nulls | Low |
| 6 | readings | Sparse 5-minute telemetry | 12,568 of 3,317,641 expected intervals present (99.62% missing) | P8: per-meter span / 5 minutes; no same-meter/same-interval conflicts | Medium |
| 7 | meters | `tariff_code` does not match tariff lookup | 194 of 194 orphans (0% coverage) | P7: meters use T01/T02/T03; tariffs use TARIFF09xxxxx codes | High |
| 8 | outages | OPEN/INVESTIGATING outages already have `outage_end_ts` | 518 of 789 | P8 event-type check | Medium |
| 9 | recovery | Recovery before linked outage started | 411 of 411 | P8: `event_ts < outage_start_ts`; the link looks off by one outage | High |
| 10 | recovery | Recovery linked to unresolved outage | 270 of 411 | P8 status transition check | Medium |
| 11 | recovery | Meter substation differs from outage substation | 404 of 411 | P8: `meter_master.substation_id` vs outage substation | Medium |
| 12 | recovery | `acknowledged_flag` lost in Bronze column | Resolved 4 Nov 2026: was 411 of 411 NULL with values only in `_rescued_data`; now Y 230, N 181, 0 rescued | Ingestion schema changed from BOOLEAN to STRING; table rebuilt once through checkpoint `recovery_v2` | Resolved |
| 13 | meters, tariffs | Trailing carriage return in last text column | 194 and 212 rows (all) | P2: `\r` in raw text (`tariff_code`, `regulatory_zone`); values parse after trim. Substations now 0: its last column `longitude` is cast to DOUBLE at ingestion | Low |
| 14 | meters, substations, tariffs | Numbers and dates stored as text | Resolved 4 Nov 2026: 11 columns now DOUBLE/DATE, 0 cast failures, row counts unchanged (194, 203, 212) | Azure SQL ingestion casts to contract types; existing tables migrated in place. Flags (`active_flag`) remain Y/N text by design | Resolved |

No problems found:

* No null or duplicate keys in outages, recovery, meters, substations or tariffs.
* Every timestamp parses.
* Interval duration is always 5 minutes.
* `received_ts` is never before the event.
* Outage end and reported times are never before start.
* Tariff expiry is never before effective.
* Coordinates are in range.
* Every foreign key other than `tariff_code` has 100% coverage.
* Voltage, current, power factor, signal strength (-104.5 to -40.6 dBm), affected meters and master-data numerics have no outliers.

Open points against the Specification and Day-wise Guide:

* The Guide expects 11,540 rows, 40 exact duplicates, 12 optional nulls and 35 extremes.
* The delivered data shows 12,617 rows, 49 duplicates, 13 nulls and 45 extremes.
* The Handbook (§4) says not to infer defect counts, so the observed values stand. Confirm with the trainer which dataset version applies.
* Finding 4 is verified by the P8 rule only. Confirm whether `installation_date` is reliable before using it as a Silver rule.